# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [32]:
!pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [33]:
import pandas as pd

trans = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rews = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = sorted(set(trans["state"]).union(set(trans["next_state"])))

#    P[(state, action)] = [(next_state, prob), (next_state, prob), ...]
P = {}
for i in range(len(trans)):
    s = trans.loc[i, "state"]
    a = trans.loc[i, "action"]
    sp = trans.loc[i, "next_state"]
    p = float(trans.loc[i, "probability"])

    key = (s, a)
    if key not in P:
        P[key] = []
    P[key].append((sp, p))

#    R[(state, action)] = reward
R = {}
for i in range(len(rews)):
    a = rews.loc[i, "action"]
    s = rews.loc[i, "state"]
    r = float(rews.loc[i, "reward"])
    R[(s, a)] = r

DO_NOTHING = "do-nothing"

gamma = 1.0      # no discount
tol = 1e-10
max_iters = 200000

# start values at 0
V = {}
for s in states:
    V[s] = 0.0

for it in range(max_iters):
    V_new = {}
    biggest_change = 0.0

    for s in states:
        a = DO_NOTHING

        # immediate reward for (s, a)
        r = R.get((s, a), 0.0)

        # expected value of next state
        outcomes = P.get((s, a), [])
        expected_next = 0.0
        for (sp, p) in outcomes:
            expected_next += p * V[sp]

        V_new[s] = r + gamma * expected_next

        change = abs(V_new[s] - V[s])
        if change > biggest_change:
            biggest_change = change

    V = V_new

    if biggest_change < tol:
        # print("Converged in", it + 1, "iterations")
        break

print("Policy = always:", DO_NOTHING)
print()
print("state\t\tV(s) = expected total reward\tExpected total discomfort")

for s in states:
    discomfort = -V[s]
    print(f"{s:10s}\t{V[s]: .6f}\t\t\t{discomfort: .6f}")

Policy = always: do-nothing

state		V(s) = expected total reward	Expected total discomfort
exposed-1 	-3.413333			 3.413333
exposed-2 	-4.266667			 4.266667
exposed-3 	-5.333333			 5.333333
recovered 	 0.000000			-0.000000
symptoms-1	-6.666667			 6.666667
symptoms-2	-5.000000			 5.000000
symptoms-3	-1.666667			 1.666667


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [34]:
# YOUR CHANGES HERE

rows = []

for s in states:
    discomfort = -V[s]
    rows.append({
        "state": s,
        "expected_discomfort": discomfort
    })

df_out = pd.DataFrame(rows)

df_out.to_csv("do-nothing-discomfort.tsv", sep="\t", index=False)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [35]:
# YOUR CHANGES HERE

trans = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rews = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = sorted(set(trans["state"]).union(set(trans["next_state"])))
actions = sorted(set(trans["action"]))

P = {}
for i in range(len(trans)):
    s = trans.loc[i, "state"]
    a = trans.loc[i, "action"]
    sp = trans.loc[i, "next_state"]
    p = float(trans.loc[i, "probability"])

    key = (s, a)
    if key not in P:
        P[key] = []
    P[key].append((sp, p))

R = {}
for i in range(len(rews)):
    s = rews.loc[i, "state"]
    a = rews.loc[i, "action"]
    r = float(rews.loc[i, "reward"])
    R[(s, a)] = r

gamma = 1.0
tol = 1e-10
max_iters = 200000

V = {}
for s in states:
    V[s] = 0.0

for it in range(max_iters):
    V_new = {}
    biggest_change = 0.0

    for s in states:
        best_value = None

        for a in actions:
            r = R.get((s, a), 0.0)

            expected_next = 0.0
            for (sp, prob) in P.get((s, a), []):
                expected_next += prob * V[sp]

            q = r + gamma * expected_next

            if (best_value is None) or (q > best_value):
                best_value = q

        V_new[s] = best_value
        biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

    V = V_new

    if biggest_change < tol:
        break

policy = {}

for s in states:
    best_action = None
    best_value = None

    for a in actions:
        r = R.get((s, a), 0.0)

        expected_next = 0.0
        for (sp, prob) in P.get((s, a), []):
            expected_next += prob * V[sp]

        q = r + gamma * expected_next

        if (best_value is None) or (q > best_value):
            best_value = q
            best_action = a

    policy[s] = best_action

print("Optimal Treatment Plan:")
for s in states:
    print(s, "->", policy[s])

Optimal Treatment Plan:
exposed-1 -> sleep-8
exposed-2 -> sleep-8
exposed-3 -> sleep-8
recovered -> do-nothing
symptoms-1 -> drink-oj
symptoms-2 -> drink-oj
symptoms-3 -> drink-oj


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [36]:
# YOUR CHANGES HERE

rows = []

for s in states:
    rows.append({
        "state": s,
        "action": policy[s]
    })

df_out = pd.DataFrame(rows)
df_out.to_csv("minimum-discomfort-actions.tsv", sep="\t", index=False)


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [37]:
# YOUR CHANGES HERE

trans = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rews = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

states = sorted(set(trans["state"]).union(set(trans["next_state"])))
actions = sorted(set(trans["action"]))

P = {}
for i in range(len(trans)):
    s = trans.loc[i, "state"]
    a = trans.loc[i, "action"]
    sp = trans.loc[i, "next_state"]
    p = float(trans.loc[i, "probability"])

    key = (s, a)
    if key not in P:
        P[key] = []
    P[key].append((sp, p))

R = {}
for i in range(len(rews)):
    s = rews.loc[i, "state"]
    a = rews.loc[i, "action"]
    r = float(rews.loc[i, "reward"])
    R[(s, a)] = r

gamma = 1.0
tol = 1e-10
max_iters = 200000


V = {}
for s in states:
    V[s] = 0.0

for it in range(max_iters):
    V_new = {}
    biggest_change = 0.0

    for s in states:
        best_value = None

        for a in actions:
            r = R.get((s, a), 0.0)

            expected_next = 0.0
            for (sp, prob) in P.get((s, a), []):
                expected_next += prob * V[sp]

            q = r + gamma * expected_next

            if (best_value is None) or (q > best_value):
                best_value = q

        V_new[s] = best_value
        biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

    V = V_new

    if biggest_change < tol:
        break

policy = {}
for s in states:
    best_action = None
    best_value = None

    for a in actions:
        r = R.get((s, a), 0.0)

        expected_next = 0.0
        for (sp, prob) in P.get((s, a), []):
            expected_next += prob * V[sp]

        q = r + gamma * expected_next

        if (best_value is None) or (q > best_value):
            best_value = q
            best_action = a

    policy[s] = best_action


V_pi = {}
for s in states:
    V_pi[s] = 0.0

for it in range(max_iters):
    V_new = {}
    biggest_change = 0.0

    for s in states:
        a = policy[s]
        r = R.get((s, a), 0.0)

        expected_next = 0.0
        for (sp, prob) in P.get((s, a), []):
            expected_next += prob * V_pi[sp]

        V_new[s] = r + gamma * expected_next
        biggest_change = max(biggest_change, abs(V_new[s] - V_pi[s]))

    V_pi = V_new

    if biggest_change < tol:
        break


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [38]:
# YOUR CHANGES HERE

rows = []
for s in states:
    rows.append({
        "state": s,
        "expected_discomfort": -V_pi[s]
    })

df_out = pd.DataFrame(rows)
df_out.to_csv("minimum-discomfort-values.tsv", sep="\t", index=False)


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [39]:
# YOUR CHANGES HERE

rews = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

duration_rews = rews.copy()

for i in range(len(duration_rews)):
    state = duration_rews.loc[i, "state"]

    if "symptoms" in state:
        duration_rews.loc[i, "reward"] = -1
    else:
        duration_rews.loc[i, "reward"] = 0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [40]:
# YOUR CHANGES HERE

duration_rews.to_csv("duration-rewards.tsv", sep="\t", index=False)


Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [41]:
# YOUR CHANGES HERE


trans = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rews = pd.read_csv("duration-rewards.tsv", sep="\t")

# States (from transitions)
states = list(trans["state"])
for s in trans["next_state"]:
    if s not in states:
        states.append(s)

actions = list(trans["action"].unique())

# (This also makes sure we only output states that are actually in the rewards file)
reward_states_in_order = list(rews["state"].unique())

P = {}
for i in range(len(trans)):
    s = trans.loc[i, "state"]
    a = trans.loc[i, "action"]
    sp = trans.loc[i, "next_state"]
    p = float(trans.loc[i, "probability"])

    key = (s, a)
    if key not in P:
        P[key] = []
    P[key].append((sp, p))

R = {}
for i in range(len(rews)):
    s = rews.loc[i, "state"]
    a = rews.loc[i, "action"]
    r = float(rews.loc[i, "reward"])
    R[(s, a)] = r

gamma = 1.0
tol = 1e-10
max_iters = 200000

V = {}
for s in states:
    V[s] = 0.0

for it in range(max_iters):
    V_new = {}
    biggest_change = 0.0

    for s in states:
        best_value = None

        for a in actions:
            r = R.get((s, a), 0.0)

            expected_next = 0.0
            for (sp, prob) in P.get((s, a), []):
                expected_next += prob * V[sp]

            q = r + gamma * expected_next

            if (best_value is None) or (q > best_value):
                best_value = q

        V_new[s] = best_value
        biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

    V = V_new

    if biggest_change < tol:
        break

policy = {}

for s in states:
    best_action = None
    best_value = None

    for a in actions:
        r = R.get((s, a), 0.0)

        expected_next = 0.0
        for (sp, prob) in P.get((s, a), []):
            expected_next += prob * V[sp]

        q = r + gamma * expected_next

        if (best_value is None) or (q > best_value):
            best_value = q
            best_action = a

    policy[s] = best_action

Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [42]:
# YOUR CHANGES HERE

rows = []
for s in reward_states_in_order:
    rows.append({"state": s, "action": policy[s]})

df_out = pd.DataFrame(rows)
df_out.to_csv("minimum-duration-actions.tsv", sep="\t", index=False)

Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [43]:
# YOUR CHANGES HERE

trans = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rews = pd.read_csv("duration-rewards.tsv", sep="\t")

policy_df = pd.read_csv("minimum-duration-actions.tsv", sep="\t")

policy = {}
for i in range(len(policy_df)):
    s = policy_df.loc[i, "state"]
    a = policy_df.loc[i, "action"]
    policy[s] = a

states = list(rews["state"].unique())

P = {}
for i in range(len(trans)):
    s = trans.loc[i, "state"]
    a = trans.loc[i, "action"]
    sp = trans.loc[i, "next_state"]
    p = float(trans.loc[i, "probability"])

    key = (s, a)
    if key not in P:
        P[key] = []
    P[key].append((sp, p))

R = {}
for i in range(len(rews)):
    s = rews.loc[i, "state"]
    a = rews.loc[i, "action"]
    r = float(rews.loc[i, "reward"])
    R[(s, a)] = r

gamma = 1.0
tol = 1e-10
max_iters = 200000

V = {}
for s in states:
    V[s] = 0.0

for it in range(max_iters):
    V_new = {}
    biggest_change = 0.0

    for s in states:
        a = policy[s]
        r = R.get((s, a), 0.0)

        expected_next = 0.0
        for (sp, prob) in P.get((s, a), []):
            expected_next += prob * V[sp]

        V_new[s] = r + gamma * expected_next
        biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

    V = V_new

    if biggest_change < tol:
        break

rows = []
for s in states:
    rows.append({
        "state": s,
        "expected_sick_days": -V[s]
    })

Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [44]:
# YOUR CHANGES HERE

df_out = pd.DataFrame(rows)
df_out.to_csv("minimum-duration-days.tsv", sep="\t", index=False)


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

In [45]:
# YOUR CHANGES HERE

trans = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")

rews = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

speed_policy_df = pd.read_csv("minimum-duration-actions.tsv", sep="\t")
comfort_policy_df = pd.read_csv("minimum-discomfort-actions.tsv", sep="\t")

speed_policy = {}
for i in range(len(speed_policy_df)):
    s = speed_policy_df.loc[i, "state"]
    a = speed_policy_df.loc[i, "action"]
    speed_policy[s] = a

comfort_policy = {}
for i in range(len(comfort_policy_df)):
    s = comfort_policy_df.loc[i, "state"]
    a = comfort_policy_df.loc[i, "action"]
    comfort_policy[s] = a

states = list(rews["state"].unique())

P = {}
for i in range(len(trans)):
    s = trans.loc[i, "state"]
    a = trans.loc[i, "action"]
    sp = trans.loc[i, "next_state"]
    p = float(trans.loc[i, "probability"])

    key = (s, a)
    if key not in P:
        P[key] = []
    P[key].append((sp, p))

R = {}
for i in range(len(rews)):
    s = rews.loc[i, "state"]
    a = rews.loc[i, "action"]
    r = float(rews.loc[i, "reward"])
    R[(s, a)] = r

def evaluate_policy(policy):
    gamma = 1.0
    tol = 1e-10
    max_iters = 200000

    V = {}
    for s in states:
        V[s] = 0.0

    for it in range(max_iters):
        V_new = {}
        biggest_change = 0.0

        for s in states:
            a = policy[s]
            r = R.get((s, a), 0.0)

            expected_next = 0.0
            for (sp, prob) in P.get((s, a), []):
                expected_next += prob * V[sp]

            V_new[s] = r + gamma * expected_next
            biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

        V = V_new

        if biggest_change < tol:
            break

    return V

V_speed = evaluate_policy(speed_policy)
V_comfort = evaluate_policy(comfort_policy)



Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [46]:
# YOUR CHANGES HERE

rows = []
for s in states:
    rows.append({
        "state": s,
        "speed_discomfort": -V_speed[s],
        "minimize_discomfort": -V_comfort[s]
    })

df_out = pd.DataFrame(rows)
df_out.to_csv("policy-comparison.tsv", sep="\t", index=False)


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [48]:
content = """I used Chat GPT to help me understand what each question was asking. It calls me Cadence becuase I share an account with my brother. https://chatgpt.com/share/69a52b69-8e40-800e-97f9-93e7caa3e795
"""

with open("acknowledgments.txt", "w") as f:
    f.write(content)